In [0]:
%sql
-- Tell the session to focus explicitly on our project catalog
USE CATALOG dbw_gitarchive;

-- Show the schemas inside to confirm 'bronze' is listening
SHOW SCHEMAS;

### Preparing to ingest the data

In [0]:
import os
import datetime
import urllib.request
from pyspark.sql.functions import current_timestamp

In [0]:
target_date = "2026-05-01"
target_hours = ["00", "01"]
print(f"Verified target hours: {target_hours}")

print(f"Targeting GitHub Archive ingestion for: {target_date} hours {target_hours}")

In [0]:
%sql
DROP TABLE IF EXISTS dbw_gitarchive.bronze.raw_github_events;

In [0]:
import urllib.request
import gzip
import json
import io
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import StructType, StructField, StringType

# --- 1. SETUP DATABRICKS WIDGETS ---
dbutils.widgets.text("date", "2024-01-01", "Target Date (YYYY-MM-DD)")
dbutils.widgets.text("hours", "00,01", "Target Hours (Comma-separated, e.g., 00,01,02)")

target_date = dbutils.widgets.get("date").strip()
hours_raw = dbutils.widgets.get("hours").strip()
target_hours = [h.strip() for h in hours_raw.split(",") if h.strip()]

target_catalog = "dbw_gitarchive"
target_schema = "bronze"
target_table = "raw_github_events"
full_table_path = f"{target_catalog}.{target_schema}.{target_table}"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

bronze_schema = StructType([
    StructField("id",           StringType(), True),
    StructField("type",         StringType(), True),
    StructField("actor",        StringType(), True),
    StructField("repo",         StringType(), True),
    StructField("payload",      StringType(), True),
    StructField("public",       StringType(), True),
    StructField("created_at",   StringType(), True),
    StructField("org",          StringType(), True),
    StructField("source_file",  StringType(), True),
])

def flatten_record(record, file_name):
    return {
        "id":          str(record.get("id", "")),
        "type":        record.get("type", ""),
        "actor":       json.dumps(record.get("actor", {})),
        "repo":        json.dumps(record.get("repo", {})),
        "payload":     json.dumps(record.get("payload", {})),
        "public":      str(record.get("public", "")),
        "created_at":  record.get("created_at", ""),
        "org":         json.dumps(record.get("org")) if record.get("org") else None,
        "source_file": file_name,
    }

# --- 2. PARAMETERIZED INGESTION LOOP ---
print(f"🎬 Starting Pipeline for Date: {target_date} | Hours to process: {target_hours}")

for hour in target_hours:
    hour_int = int(hour)
    file_name = f"{target_date}-{hour_int}.json.gz"
    source_url = f"https://data.gharchive.org/{file_name}"
    
    print(f"🚀 Ingesting: {source_url}")
    
    try:
        request = urllib.request.Request(source_url, headers=headers)
        with urllib.request.urlopen(request) as response:
            gz_bytes = response.read()

        records = []
        with gzip.open(io.BytesIO(gz_bytes), 'rt', encoding='utf-8') as gz_file:
            for line in gz_file:
                if line.strip():
                    records.append(flatten_record(json.loads(line), file_name))

        if not records:
            print(f"  ⚠️ No records found for hour {hour_int}. Skipping.")
            continue

        # 1. We create the data and give it the 'event_date' column
        df_bronze = spark.createDataFrame(records, schema=bronze_schema) \
                         .withColumn("ingested_at", current_timestamp()) \
                         .withColumn("event_date", lit(target_date))

        # 2. We overwrite just this date. Since the table is brand new, it will accept it flawlessly!
        df_bronze.write \
            .format("delta") \
            .mode("append") \
            .option("replaceWhere", f"event_date = '{target_date}'") \
            .option("mergeSchema", "true") \
            .saveAsTable(full_table_path)

        print(f"  ✅ Hour {hour_int:02d} done — {len(records):,} rows written.\n")

    except Exception as e:
        print(f"  ❌ Failed on hour {hour_int:02d}: {e}\n")

In [0]:
%sql
SHOW SCHEMAS IN dbw_gitarchive;

In [0]:
%sql
SELECT type, actor, repo, payload, event_date 
FROM dbw_gitarchive.bronze.raw_github_events 
LIMIT 10;

In [0]:
%sql
select COUNT(*) FROM dbw_gitarchive.bronze.raw_github_events;